In [1]:
import pandas as pd
import numpy as np

In [2]:
from _counties import get_counties_df
counties_df = get_counties_df()
counties_df

,CO_FIPS,CO_NAME
0,5,CACHE
1,13,DUCHESNE
2,11,DAVIS
3,27,MILLARD
4,51,WASATCH
5,3,BOX ELDER
6,57,WEBER
7,23,JUAB
8,39,SANPETE
9,53,WASHINGTON


In [3]:
# read excel file into df
file_path = r"data\GPI-Base-Data-20251201\2023_2024_UDEM_calibration_data_20251201.xlsx"

# Read **first** sheet (sheet_name=0)
base_year_2023_df = pd.read_excel(file_path, sheet_name=0, usecols=['Year','Geography','POP','HH','GQ','HHPOP']).rename(columns={'Year':'YEAR'})

base_year_2023_df['CO_NAME'] = base_year_2023_df['Geography'].str.replace(' COUNTY', '')
base_year_2023_df['CO_NAME'] = base_year_2023_df['CO_NAME'].str.upper()

base_year_2023_df.drop(columns=['Geography'], inplace=True)

base_year_2023_df = base_year_2023_df.merge(counties_df, on='CO_NAME')

#base_year_2023_df['YEAR'] = 2023

display(base_year_2023_df)


,YEAR,POP,HH,GQ,HHPOP,CO_NAME,CO_FIPS
0,2023,7.313585e+03,2381.171904,299.428835,7.014156e+03,BEAVER,1
1,2023,6.125078e+04,20602.544283,166.019149,6.108476e+04,BOX ELDER,3
2,2023,1.416997e+05,45994.953039,4463.412384,1.372363e+05,CACHE,5
3,2023,2.065354e+04,7948.667611,116.045823,2.053749e+04,CARBON,7
4,2023,9.977325e+02,420.750519,0.000000,9.977325e+02,DAGGETT,9
5,2023,3.773785e+05,122330.094345,2404.727933,3.749737e+05,DAVIS,11
6,2023,2.011194e+04,6777.016795,87.471292,2.002446e+04,DUCHESNE,13
7,2023,1.003529e+04,3794.104779,58.000000,9.977291e+03,EMERY,15
8,2023,5.140647e+03,1963.103222,204.163255,4.936484e+03,GARFIELD,17
9,2023,9.842434e+03,4170.780935,156.000000,9.686434e+03,GRAND,19


In [4]:
gpi_year_age_df = pd.read_csv("data/GPI-SE-Data-20251121/utah_proj_v2025_sya_2020_2065.csv").rename(columns={'Year':'YEAR'})
gpi_2025_age_df = gpi_year_age_df[gpi_year_age_df['YEAR'].isin([2023,2024])]
gpi_2025_age_df

,Geography,YEAR,Age,Pop
303,Beaver County,2023,0,103
304,Beaver County,2023,1,114
305,Beaver County,2023,2,95
306,Beaver County,2023,3,110
307,Beaver County,2023,4,110
...,...,...,...,...
135234,State of Utah,2024,96,846
135235,State of Utah,2024,97,580
135236,State of Utah,2024,98,401
135237,State of Utah,2024,99,251


In [5]:
# df columns expected:
# Geography | Year | Age | Pop

# 1. Define your age-group bins
bins = [0, 17, 64, 200]          # max age large enough to include all ages
labels = ["POP_00_17_pct", "POP_18_64_pct", "POP_65P_pct"]

# 2. Assign each record to an age group
gpi_2025_age_df["AgeGroup"] = pd.cut(gpi_2025_age_df["Age"].astype(int), bins=bins, labels=labels, right=True)

# 3. Aggregate
gpi_2025_age_groups_df = (
    gpi_2025_age_df.groupby(["Geography", "YEAR", "AgeGroup"], as_index=False, observed=False)
      .agg({"Pop": "sum"})
      .rename(columns={"AgeGroup": "Age"})
)

gpi_2025_age_groups_df = gpi_2025_age_groups_df[gpi_2025_age_groups_df['Geography'].str.contains(' County')]

gpi_2025_age_groups_df['CO_NAME'] = gpi_2025_age_groups_df['Geography'].str.replace(' County','').str.upper()

gpi_2025_age_groups_df.drop(columns=['Geography'], inplace=True)

display(gpi_2025_age_groups_df)


C:\Users\bhereth\AppData\Local\Temp\ipykernel_28568\3606837317.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gpi_2025_age_df["AgeGroup"] = pd.cut(gpi_2025_age_df["Age"].astype(int), bins=bins, labels=labels, right=True)


,YEAR,Age,Pop,CO_NAME
0,2023,POP_00_17_pct,1992,BEAVER
1,2023,POP_18_64_pct,4129,BEAVER
2,2023,POP_65P_pct,1095,BEAVER
3,2024,POP_00_17_pct,1969,BEAVER
4,2024,POP_18_64_pct,4151,BEAVER
...,...,...,...,...
175,2023,POP_18_64_pct,165068,WEBER
176,2023,POP_65P_pct,34713,WEBER
177,2024,POP_00_17_pct,65593,WEBER
178,2024,POP_18_64_pct,166459,WEBER


In [6]:
# get percent of total in each Age Category
# 1. Compute total population by CO_NAME + Year
totals = (
    gpi_2025_age_groups_df
        .groupby(["CO_NAME", "YEAR"], as_index=False)
        .agg(TotalPop=("Pop", "sum"))
)

# 2. Merge totals back to the age-group DataFrame
gpi_2025_age_dist_df = gpi_2025_age_groups_df.merge(totals, on=["CO_NAME", "YEAR"])

# 3. Compute percent of total
gpi_2025_age_dist_df["PctOfTotal"] = gpi_2025_age_dist_df["Pop"] / gpi_2025_age_dist_df["TotalPop"]

gpi_2025_age_dist_df = (
    gpi_2025_age_dist_df[["CO_NAME", "YEAR", "Age", "PctOfTotal"]]
        .pivot(index=["CO_NAME",'YEAR'], columns="Age", values="PctOfTotal")
        .reset_index()
)

gpi_2025_age_dist_df

Age,CO_NAME,YEAR,POP_00_17_pct,POP_18_64_pct,POP_65P_pct
0,BEAVER,2023,0.276053,0.572201,0.151746
1,BEAVER,2024,0.272112,0.573659,0.154229
2,BOX ELDER,2023,0.287059,0.576945,0.135996
3,BOX ELDER,2024,0.280810,0.579138,0.140052
4,CACHE,2023,0.267763,0.628493,0.103743
5,CACHE,2024,0.259574,0.635060,0.105366
6,CARBON,2023,0.243869,0.562681,0.193450
7,CARBON,2024,0.239573,0.563303,0.197124
8,DAGGETT,2023,0.204431,0.534743,0.260826
9,DAGGETT,2024,0.192464,0.537678,0.269857


# Apply 2025 split to 2023

In [7]:
pop_df = pd.merge(base_year_2023_df, gpi_2025_age_dist_df, on=('CO_NAME','YEAR'))

pop_df['TOTPOP'] = np.round(pop_df['POP']).astype(int)
pop_df['HHPOP'] = np.round(pop_df['HHPOP']).astype(int)
pop_df['GQPOP'] = pop_df['TOTPOP'] - pop_df['HHPOP']

pop_df['POP0017'] = np.round(pop_df['POP'] * pop_df['POP_00_17_pct']).astype(int)
pop_df['POP1864'] = np.round(pop_df['POP'] * pop_df['POP_18_64_pct']).astype(int)
pop_df['POP65P'] = pop_df['TOTPOP'] - pop_df['POP0017'] - pop_df['POP1864']

pop_df = pop_df[['CO_FIPS','YEAR','TOTPOP','GQPOP','HHPOP','POP0017','POP1864','POP65P','HH','CO_NAME']]

pop_melt_df = pop_df.melt(id_vars=['CO_FIPS','YEAR','CO_NAME'], var_name='TDM_VARIABLE', value_name='VALUE')
pop_melt_df['VALUE'] = pop_melt_df['VALUE'].astype(int)
pop_melt_df

,CO_FIPS,YEAR,CO_NAME,TDM_VARIABLE,VALUE
0,1,2023,BEAVER,TOTPOP,7314
1,3,2023,BOX ELDER,TOTPOP,61251
2,5,2023,CACHE,TOTPOP,141700
3,7,2023,CARBON,TOTPOP,20654
4,9,2023,DAGGETT,TOTPOP,998
...,...,...,...,...,...
401,49,2024,UTAH,HH,228987
402,51,2024,WASATCH,HH,13642
403,53,2024,WASHINGTON,HH,75803
404,55,2024,WAYNE,HH,960


In [8]:
pop_melt_df[['CO_FIPS','YEAR','TDM_VARIABLE','VALUE','CO_NAME']].to_csv('intermediate/tdm_population_by_county_2023_2024.csv', index=False)